In [ ]:
"""
================================================================================
BirdCLEF+ 2026 - v8 Training
================================================================================

Pipeline: Perch -> ProtoSSM(pass1) + MLP -> ensemble -> ResidualSSM(pass2) -> final

Key Components:
1. Perch v2 embeddings with genus-level proxy mapping for unmapped species
2. ProtoSSM v2: State Space Model with prototypical classification
   - Metadata embeddings (site/hour)
   - Knowledge distillation from Perch
   - Per-class gated fusion with Perch logits
3. MLP probes on PCA-compressed embeddings
4. ResidualSSM: Second-pass error correction on first-pass residuals
5. Prior tables for site/hour/site-hour metadata
6. Temporal smoothing (texture: moving avg, event: local max blend)
7. OOF cross-validation with GroupKFold

Output Files:
  - full_perch_arrays.npz    → embeddings + scores + models + indices
  - full_perch_meta.parquet  → metadata

================================================================================
"""

# =============================================================================
# INSTALL TENSORFLOW 2.20 (Required for Perch v2)
# =============================================================================
!pip install -q --no-deps /kaggle/input/notebooks/kdmitrie/bc26-tensorflow-2-20-0/wheel/tensorboard-2.20.0-py3-none-any.whl
!pip install -q --no-deps /kaggle/input/notebooks/kdmitrie/bc26-tensorflow-2-20-0/wheel/tensorflow-2.20.0-cp312-cp312-manylinux_2_17_x86_64.manylinux2014_x86_64.whl

import os
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"
os.environ["CUDA_VISIBLE_DEVICES"] = ""

import gc
import json
import re
import time
import warnings
from collections import defaultdict
from io import BytesIO
from pathlib import Path

import numpy as np
import pandas as pd
import soundfile as sf
import tensorflow as tf

import torch
import torch.nn as nn
import torch.nn.functional as F

from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import GroupKFold
from sklearn.preprocessing import StandardScaler

from tqdm.auto import tqdm
import pickle

warnings.filterwarnings("ignore")
tf.experimental.numpy.experimental_enable_numpy_behavior()

# =============================================================================
# CONFIGURATION
# =============================================================================
BASE = Path("/kaggle/input/competitions/birdclef-2026")
MODEL_DIR = Path("/kaggle/input/models/google/bird-vocalization-classifier/tensorflow2/perch_v2_cpu/1")
OUTPUT_DIR = Path("/kaggle/working")

SR = 32000
WINDOW_SEC = 5
WINDOW_SAMPLES = SR * WINDOW_SEC
FILE_SAMPLES = 60 * SR
N_WINDOWS = 12

DEVICE = torch.device("cpu")
_WALL_START = time.time()
LOGS = {}

CFG = {
    "verbose": True,
    "batch_files": 16,
    "proxy_reduce": "max",

    # Frozen baseline fusion params
    "best_fusion": {
        "lambda_event": 0.4,
        "lambda_texture": 1.0,
        "lambda_proxy_texture": 0.8,
        "smooth_texture": 0.35,
        "smooth_event": 0.15,
    },

    # ProtoSSM v2 architecture
    "proto_ssm": {
        "d_model": 192,
        "d_state": 16,
        "n_ssm_layers": 2,
        "dropout": 0.15,
        "n_prototypes": 1,
        "n_sites": 20,
        "meta_dim": 16,
    },

    # ProtoSSM v2 training
    "proto_ssm_train": {
        "n_epochs": 60,
        "lr": 1e-3,
        "weight_decay": 2e-3,
        "val_ratio": 0.15,
        "patience": 15,
        "pos_weight_cap": 30.0,
        "distill_weight": 0.1,
        "proto_margin": 0.1,
        "label_smoothing": 0.02,
        "oof_n_splits": 3,
    },

    # Frozen probe params
    "frozen_best_probe": {
        "pca_dim": 64,
        "min_pos": 8,
        "C": 0.50,
        "alpha": 0.40,
    },

    # Residual SSM (second pass boosting)
    "residual_ssm": {
        "d_model": 64,
        "d_state": 8,
        "n_ssm_layers": 1,
        "dropout": 0.1,
        "correction_weight": 0.3,
        "n_epochs": 30,
        "lr": 1e-3,
        "patience": 8,
    },

    "probe_backend": "mlp",
    "mlp_params": {
        "hidden_layer_sizes": (128,),
        "activation": "relu",
        "max_iter": 300,
        "early_stopping": True,
        "validation_fraction": 0.15,
        "n_iter_no_change": 15,
        "random_state": 42,
        "learning_rate_init": 0.001,
        "alpha": 0.01,
    },
}

BEST = CFG["best_fusion"]

print("=" * 70)
print("BirdCLEF+ 2026 - v8 Training")
print("=" * 70)
print("TensorFlow:", tf.__version__)
print("PyTorch:", torch.__version__)
print("Device:", DEVICE)

# =============================================================================
# LOAD DATA
# =============================================================================
print("\nLoading Data...")

taxonomy = pd.read_csv(BASE / "taxonomy.csv")
sample_sub = pd.read_csv(BASE / "sample_submission.csv")
soundscape_labels = pd.read_csv(BASE / "train_soundscapes_labels.csv")

PRIMARY_LABELS = sample_sub.columns[1:].tolist()
N_CLASSES = len(PRIMARY_LABELS)

taxonomy["primary_label"] = taxonomy["primary_label"].astype(str)
soundscape_labels["primary_label"] = soundscape_labels["primary_label"].astype(str)

label_to_idx = {c: i for i, c in enumerate(PRIMARY_LABELS)}

print(f"Classes: {N_CLASSES}")

# =============================================================================
# PARSE LABELS
# =============================================================================
def parse_soundscape_labels(x):
    if pd.isna(x):
        return []
    return [t.strip() for t in str(x).split(";") if t.strip()]


def union_labels(series):
    return sorted(set(lbl for x in series for lbl in parse_soundscape_labels(x)))


FNAME_RE = re.compile(r"BC2026_(?:Train|Test)_(\d+)_(S\d+)_(\d{8})_(\d{6})\.ogg")


def parse_soundscape_filename(name):
    m = FNAME_RE.match(name)
    if not m:
        return {
            "file_id": None,
            "site": None,
            "date": pd.NaT,
            "time_utc": None,
            "hour_utc": -1,
            "month": -1,
        }
    file_id, site, ymd, hms = m.groups()
    dt = pd.to_datetime(ymd, format="%Y%m%d", errors="coerce")
    return {
        "file_id": file_id,
        "site": site,
        "date": dt,
        "time_utc": hms,
        "hour_utc": int(hms[:2]),
        "month": int(dt.month) if pd.notna(dt) else -1,
    }


# Deduplicate and aggregate labels
sc_clean = (
    soundscape_labels
    .groupby(["filename", "start", "end"])["primary_label"]
    .apply(union_labels)
    .reset_index(name="label_list")
)

sc_clean["start_sec"] = pd.to_timedelta(sc_clean["start"]).dt.total_seconds().astype(int)
sc_clean["end_sec"] = pd.to_timedelta(sc_clean["end"]).dt.total_seconds().astype(int)
sc_clean["row_id"] = sc_clean["filename"].str.replace(".ogg", "", regex=False) + "_" + sc_clean["end_sec"].astype(str)

meta = sc_clean["filename"].apply(parse_soundscape_filename).apply(pd.Series)
sc_clean = pd.concat([sc_clean, meta], axis=1)

# Fully-labeled files
windows_per_file = sc_clean.groupby("filename").size()
full_files = sorted(windows_per_file[windows_per_file == N_WINDOWS].index.tolist())
sc_clean["file_fully_labeled"] = sc_clean["filename"].isin(full_files)

# Multi-hot label matrix
Y_SC = np.zeros((len(sc_clean), N_CLASSES), dtype=np.uint8)
for i, labels in enumerate(sc_clean["label_list"]):
    idxs = [label_to_idx[lbl] for lbl in labels if lbl in label_to_idx]
    if idxs:
        Y_SC[i, idxs] = 1

full_truth = (
    sc_clean[sc_clean["file_fully_labeled"]]
    .sort_values(["filename", "end_sec"])
    .reset_index(drop=False)
)

Y_FULL_TRUTH = Y_SC[full_truth["index"].to_numpy()]

print("sc_clean:", sc_clean.shape)
print("Y_SC:", Y_SC.shape, Y_SC.dtype)
print("Full files:", len(full_files))
print("Trusted full windows:", len(full_truth))
print("Active classes in full windows:", int((Y_FULL_TRUTH.sum(axis=0) > 0).sum()))

# =============================================================================
# LOAD PERCH AND BUILD MAPPING
# =============================================================================
print("\nLoading Perch model...")

birdclassifier = tf.saved_model.load(str(MODEL_DIR))
infer_fn = birdclassifier.signatures["serving_default"]

bc_labels = (
    pd.read_csv(MODEL_DIR / "assets" / "labels.csv")
    .reset_index()
    .rename(columns={"index": "bc_index", "inat2024_fsd50k": "scientific_name"})
)

NO_LABEL_INDEX = len(bc_labels)

# Build mapping
MANUAL_SCIENTIFIC_NAME_MAP = {}

taxonomy = taxonomy.copy()
taxonomy["scientific_name_lookup"] = taxonomy["scientific_name"].replace(MANUAL_SCIENTIFIC_NAME_MAP)

bc_lookup = bc_labels.rename(columns={"scientific_name": "scientific_name_lookup"})

mapping = taxonomy.merge(
    bc_lookup[["scientific_name_lookup", "bc_index"]],
    on="scientific_name_lookup",
    how="left"
)

mapping["bc_index"] = mapping["bc_index"].fillna(NO_LABEL_INDEX).astype(int)

label_to_bc_index = mapping.set_index("primary_label")["bc_index"]
BC_INDICES = np.array([int(label_to_bc_index.loc[c]) for c in PRIMARY_LABELS], dtype=np.int32)

MAPPED_MASK = BC_INDICES != NO_LABEL_INDEX
MAPPED_POS = np.where(MAPPED_MASK)[0].astype(np.int32)
UNMAPPED_POS = np.where(~MAPPED_MASK)[0].astype(np.int32)
MAPPED_BC_INDICES = BC_INDICES[MAPPED_MASK].astype(np.int32)

# Taxonomic groups
CLASS_NAME_MAP = taxonomy.set_index("primary_label")["class_name"].to_dict()
TEXTURE_TAXA = {"Amphibia", "Insecta"}

ACTIVE_CLASSES = [PRIMARY_LABELS[i] for i in np.where(Y_SC.sum(axis=0) > 0)[0]]

idx_active_texture = np.array(
    [label_to_idx[c] for c in ACTIVE_CLASSES if CLASS_NAME_MAP.get(c) in TEXTURE_TAXA],
    dtype=np.int32
)
idx_active_event = np.array(
    [label_to_idx[c] for c in ACTIVE_CLASSES if CLASS_NAME_MAP.get(c) not in TEXTURE_TAXA],
    dtype=np.int32
)

idx_mapped_active_texture = idx_active_texture[MAPPED_MASK[idx_active_texture]]
idx_mapped_active_event = idx_active_event[MAPPED_MASK[idx_active_event]]
idx_unmapped_active_texture = idx_active_texture[~MAPPED_MASK[idx_active_texture]]
idx_unmapped_active_event = idx_active_event[~MAPPED_MASK[idx_active_event]]
idx_unmapped_inactive = np.array(
    [i for i in UNMAPPED_POS if PRIMARY_LABELS[i] not in ACTIVE_CLASSES],
    dtype=np.int32
)

# Build genus proxies for unmapped non-sonotypes
unmapped_df = mapping[mapping["bc_index"] == NO_LABEL_INDEX].copy()
unmapped_non_sonotype = unmapped_df[
    ~unmapped_df["primary_label"].astype(str).str.contains("son", na=False)
].copy()


def get_genus_hits(scientific_name):
    genus = str(scientific_name).split()[0]
    hits = bc_labels[
        bc_labels["scientific_name"].astype(str).str.match(rf"^{re.escape(genus)}\s", na=False)
    ].copy()
    return genus, hits


proxy_map = {}
for _, row in unmapped_non_sonotype.iterrows():
    target = row["primary_label"]
    sci = row["scientific_name"]
    genus, hits = get_genus_hits(sci)
    if len(hits) > 0:
        proxy_map[target] = {
            "target_scientific_name": sci,
            "genus": genus,
            "bc_indices": hits["bc_index"].astype(int).tolist(),
            "proxy_scientific_names": hits["scientific_name"].tolist(),
        }

# Enable genus proxies for Amphibia, Insecta, and Aves
PROXY_TAXA = {"Amphibia", "Insecta", "Aves"}
SELECTED_PROXY_TARGETS = sorted([
    t for t in proxy_map.keys()
    if CLASS_NAME_MAP.get(t) in PROXY_TAXA
])

selected_proxy_pos = np.array([label_to_idx[c] for c in SELECTED_PROXY_TARGETS], dtype=np.int32)
selected_proxy_pos_to_bc = {
    label_to_idx[target]: np.array(proxy_map[target]["bc_indices"], dtype=np.int32)
    for target in SELECTED_PROXY_TARGETS
}

idx_selected_proxy_active_texture = np.intersect1d(selected_proxy_pos, idx_active_texture)
idx_selected_prioronly_active_texture = np.setdiff1d(idx_unmapped_active_texture, selected_proxy_pos)
idx_selected_prioronly_active_event = np.setdiff1d(idx_unmapped_active_event, selected_proxy_pos)

print(f"Mapped classes: {MAPPED_MASK.sum()} / {N_CLASSES}")
print(f"Unmapped classes: {(~MAPPED_MASK).sum()}")
print("Selected frog proxy targets:", SELECTED_PROXY_TARGETS)
print("Active texture classes:", len(idx_active_texture))
print("Selected proxy active texture:", len(idx_selected_proxy_active_texture))
print("Prior-only active texture:", len(idx_selected_prioronly_active_texture))
print("Prior-only active event:", len(idx_selected_prioronly_active_event))

# =============================================================================
# METRICS AND UTILITIES
# =============================================================================
def macro_auc_skip_empty(y_true, y_score):
    keep = y_true.sum(axis=0) > 0
    if keep.sum() == 0:
        return 0.0
    return roc_auc_score(y_true[:, keep], y_score[:, keep], average="macro")


def smooth_cols_fixed12(scores, cols, alpha=0.35):
    """Temporal smoothing for texture classes (moving average)."""
    if alpha <= 0 or len(cols) == 0:
        return scores.copy()
    s = scores.copy()
    assert len(s) % N_WINDOWS == 0
    view = s.reshape(-1, N_WINDOWS, s.shape[1])
    x = view[:, :, cols]
    prev_x = np.concatenate([x[:, :1, :], x[:, :-1, :]], axis=1)
    next_x = np.concatenate([x[:, 1:, :], x[:, -1:, :]], axis=1)
    view[:, :, cols] = (1.0 - alpha) * x + 0.5 * alpha * (prev_x + next_x)
    return s


def smooth_events_fixed12(scores, cols, alpha=0.15):
    """Soft max-pool context for event birds (Aves)."""
    if alpha <= 0 or len(cols) == 0:
        return scores.copy()
    s = scores.copy()
    assert len(s) % N_WINDOWS == 0
    view = s.reshape(-1, N_WINDOWS, s.shape[1])
    x = view[:, :, cols]
    prev_x = np.concatenate([x[:, :1, :], x[:, :-1, :]], axis=1)
    next_x = np.concatenate([x[:, 1:, :], x[:, -1:, :]], axis=1)
    local_max = np.maximum(x, np.maximum(prev_x, next_x))
    view[:, :, cols] = (1.0 - alpha) * x + alpha * local_max
    return s


def seq_features_1d(v):
    """Extract sequential features with std for temporal variance."""
    assert len(v) % N_WINDOWS == 0
    x = v.reshape(-1, N_WINDOWS)
    prev_v = np.concatenate([x[:, :1], x[:, :-1]], axis=1).reshape(-1)
    next_v = np.concatenate([x[:, 1:], x[:, -1:]], axis=1).reshape(-1)
    mean_v = np.repeat(x.mean(axis=1), N_WINDOWS)
    max_v = np.repeat(x.max(axis=1), N_WINDOWS)
    std_v = np.repeat(x.std(axis=1), N_WINDOWS)
    return prev_v, next_v, mean_v, max_v, std_v


def build_class_features(emb_proj, raw_col, prior_col, base_col):
    """Build features for probe training: embedding + 7 sequential + 3 interaction + std + 3 diff."""
    prev_base, next_base, mean_base, max_base, std_base = seq_features_1d(base_col)

    diff_mean = base_col - mean_base
    diff_prev = base_col - prev_base
    diff_next = base_col - next_base

    feats = np.concatenate([
        emb_proj,
        raw_col[:, None],
        prior_col[:, None],
        base_col[:, None],
        prev_base[:, None],
        next_base[:, None],
        mean_base[:, None],
        max_base[:, None],
        std_base[:, None],
        diff_mean[:, None],
        diff_prev[:, None],
        diff_next[:, None],
        (raw_col * prior_col)[:, None],
        (raw_col * base_col)[:, None],
        (prior_col * base_col)[:, None],
    ], axis=1)

    return feats.astype(np.float32, copy=False)

# =============================================================================
# PRIOR TABLES
# =============================================================================
def fit_prior_tables(prior_df, Y_prior):
    prior_df = prior_df.reset_index(drop=True)
    global_p = Y_prior.mean(axis=0).astype(np.float32)

    # Site
    site_keys = sorted(prior_df["site"].dropna().astype(str).unique().tolist())
    site_to_i = {k: i for i, k in enumerate(site_keys)}
    site_n = np.zeros(len(site_keys), dtype=np.float32)
    site_p = np.zeros((len(site_keys), Y_prior.shape[1]), dtype=np.float32)
    for s in site_keys:
        i = site_to_i[s]
        mask = prior_df["site"].astype(str).values == s
        site_n[i] = mask.sum()
        site_p[i] = Y_prior[mask].mean(axis=0)

    # Hour
    hour_keys = sorted(prior_df["hour_utc"].dropna().astype(int).unique().tolist())
    hour_to_i = {h: i for i, h in enumerate(hour_keys)}
    hour_n = np.zeros(len(hour_keys), dtype=np.float32)
    hour_p = np.zeros((len(hour_keys), Y_prior.shape[1]), dtype=np.float32)
    for h in hour_keys:
        i = hour_to_i[h]
        mask = prior_df["hour_utc"].astype(int).values == h
        hour_n[i] = mask.sum()
        hour_p[i] = Y_prior[mask].mean(axis=0)

    # Site-hour
    sh_to_i = {}
    sh_n_list = []
    sh_p_list = []
    for (s, h), idx in prior_df.groupby(["site", "hour_utc"]).groups.items():
        sh_to_i[(str(s), int(h))] = len(sh_n_list)
        idx = np.array(list(idx))
        sh_n_list.append(len(idx))
        sh_p_list.append(Y_prior[idx].mean(axis=0))

    sh_n = np.array(sh_n_list, dtype=np.float32)
    sh_p = np.stack(sh_p_list).astype(np.float32) if sh_p_list else np.zeros((0, Y_prior.shape[1]), dtype=np.float32)

    return {
        "global_p": global_p,
        "site_to_i": site_to_i,
        "site_n": site_n,
        "site_p": site_p,
        "hour_to_i": hour_to_i,
        "hour_n": hour_n,
        "hour_p": hour_p,
        "sh_to_i": sh_to_i,
        "sh_n": sh_n,
        "sh_p": sh_p,
    }


def prior_logits_from_tables(sites, hours, tables, eps=1e-4):
    n = len(sites)
    p = np.repeat(tables["global_p"][None, :], n, axis=0).astype(np.float32, copy=True)

    site_idx = np.fromiter(
        (tables["site_to_i"].get(str(s), -1) for s in sites),
        dtype=np.int32,
        count=n
    )
    hour_idx = np.fromiter(
        (tables["hour_to_i"].get(int(h), -1) if int(h) >= 0 else -1 for h in hours),
        dtype=np.int32,
        count=n
    )
    sh_idx = np.fromiter(
        (tables["sh_to_i"].get((str(s), int(h)), -1) if int(h) >= 0 else -1 for s, h in zip(sites, hours)),
        dtype=np.int32,
        count=n
    )

    valid = hour_idx >= 0
    if valid.any():
        nh = tables["hour_n"][hour_idx[valid]][:, None]
        wh = nh / (nh + 8.0)
        p[valid] = wh * tables["hour_p"][hour_idx[valid]] + (1.0 - wh) * p[valid]

    valid = site_idx >= 0
    if valid.any():
        ns = tables["site_n"][site_idx[valid]][:, None]
        ws = ns / (ns + 8.0)
        p[valid] = ws * tables["site_p"][site_idx[valid]] + (1.0 - ws) * p[valid]

    valid = sh_idx >= 0
    if valid.any():
        nsh = tables["sh_n"][sh_idx[valid]][:, None]
        wsh = nsh / (nsh + 4.0)
        p[valid] = wsh * tables["sh_p"][sh_idx[valid]] + (1.0 - wsh) * p[valid]

    np.clip(p, eps, 1.0 - eps, out=p)
    return (np.log(p) - np.log1p(-p)).astype(np.float32, copy=False)


def fuse_scores_with_tables(base_scores, sites, hours, tables):
    scores = base_scores.copy()
    prior = prior_logits_from_tables(sites, hours, tables)

    # mapped active
    if len(idx_mapped_active_event):
        scores[:, idx_mapped_active_event] += BEST["lambda_event"] * prior[:, idx_mapped_active_event]
    if len(idx_mapped_active_texture):
        scores[:, idx_mapped_active_texture] += BEST["lambda_texture"] * prior[:, idx_mapped_active_texture]

    # selected frog proxies
    if len(idx_selected_proxy_active_texture):
        scores[:, idx_selected_proxy_active_texture] += BEST["lambda_proxy_texture"] * prior[:, idx_selected_proxy_active_texture]

    # prior-only active unmapped
    if len(idx_selected_prioronly_active_event):
        scores[:, idx_selected_prioronly_active_event] = BEST["lambda_event"] * prior[:, idx_selected_prioronly_active_event]
    if len(idx_selected_prioronly_active_texture):
        scores[:, idx_selected_prioronly_active_texture] = BEST["lambda_texture"] * prior[:, idx_selected_prioronly_active_texture]

    # inactive unmapped
    if len(idx_unmapped_inactive):
        scores[:, idx_unmapped_inactive] = -8.0

    scores = smooth_cols_fixed12(scores, idx_active_texture, alpha=BEST["smooth_texture"])
    scores = smooth_events_fixed12(scores, idx_active_event, alpha=BEST["smooth_event"])
    return scores.astype(np.float32, copy=False), prior

# =============================================================================
# PERCH INFERENCE ENGINE
# =============================================================================
def read_soundscape_60s(path):
    y, sr = sf.read(path, dtype="float32", always_2d=False)
    if y.ndim == 2:
        y = y.mean(axis=1)
    if sr != SR:
        raise ValueError(f"Unexpected sample rate {sr}")
    if len(y) < FILE_SAMPLES:
        y = np.pad(y, (0, FILE_SAMPLES - len(y)))
    elif len(y) > FILE_SAMPLES:
        y = y[:FILE_SAMPLES]
    return y


def infer_perch_with_embeddings(paths, batch_files=16, verbose=True, proxy_reduce="max"):
    paths = [Path(p) for p in paths]
    n_files = len(paths)
    n_rows = n_files * N_WINDOWS

    row_ids = np.empty(n_rows, dtype=object)
    filenames = np.empty(n_rows, dtype=object)
    sites = np.empty(n_rows, dtype=object)
    hours = np.empty(n_rows, dtype=np.int16)

    scores = np.zeros((n_rows, N_CLASSES), dtype=np.float32)
    embeddings = np.zeros((n_rows, 1536), dtype=np.float32)

    write_row = 0
    iterator = range(0, n_files, batch_files)
    if verbose:
        iterator = tqdm(iterator, total=(n_files + batch_files - 1) // batch_files, desc="Perch batches")

    for start in iterator:
        batch_paths = paths[start:start + batch_files]
        batch_n = len(batch_paths)

        x = np.empty((batch_n * N_WINDOWS, WINDOW_SAMPLES), dtype=np.float32)
        batch_row_start = write_row
        x_pos = 0

        for path in batch_paths:
            y = read_soundscape_60s(path)
            x[x_pos:x_pos + N_WINDOWS] = y.reshape(N_WINDOWS, WINDOW_SAMPLES)

            meta = parse_soundscape_filename(path.name)
            stem = path.stem

            row_ids[write_row:write_row + N_WINDOWS] = [f"{stem}_{t}" for t in range(5, 65, 5)]
            filenames[write_row:write_row + N_WINDOWS] = path.name
            sites[write_row:write_row + N_WINDOWS] = meta["site"]
            hours[write_row:write_row + N_WINDOWS] = int(meta["hour_utc"])

            x_pos += N_WINDOWS
            write_row += N_WINDOWS

        outputs = infer_fn(inputs=tf.convert_to_tensor(x))
        logits = outputs["label"].numpy().astype(np.float32, copy=False)
        emb = outputs["embedding"].numpy().astype(np.float32, copy=False)

        scores[batch_row_start:write_row, MAPPED_POS] = logits[:, MAPPED_BC_INDICES]
        embeddings[batch_row_start:write_row] = emb

        for pos, bc_idx_arr in selected_proxy_pos_to_bc.items():
            sub = logits[:, bc_idx_arr]
            if proxy_reduce == "max":
                proxy_score = sub.max(axis=1)
            elif proxy_reduce == "mean":
                proxy_score = sub.mean(axis=1)
            else:
                raise ValueError("proxy_reduce must be 'max' or 'mean'")
            scores[batch_row_start:write_row, pos] = proxy_score.astype(np.float32)

        del x, outputs, logits, emb
        gc.collect()

    meta_df = pd.DataFrame({
        "row_id": row_ids,
        "filename": filenames,
        "site": sites,
        "hour_utc": hours,
    })

    return meta_df, scores, embeddings

# =============================================================================
# RUN PERCH ON TRAINING DATA
# =============================================================================
print("\n" + "-" * 70)
print("Running Perch on trusted full files...")
print("-" * 70)

full_paths = [BASE / "train_soundscapes" / fn for fn in full_files]
meta_full, scores_full_raw, emb_full = infer_perch_with_embeddings(
    full_paths,
    batch_files=CFG["batch_files"],
    verbose=CFG["verbose"],
    proxy_reduce=CFG["proxy_reduce"],
)

# Align truth
full_truth_aligned = full_truth.set_index("row_id").loc[meta_full["row_id"]].reset_index()
Y_FULL = Y_SC[full_truth_aligned["index"].to_numpy()]

print("meta_full:", meta_full.shape)
print("scores_full_raw:", scores_full_raw.shape, scores_full_raw.dtype)
print("emb_full:", emb_full.shape, emb_full.dtype)
print("Y_FULL:", Y_FULL.shape, Y_FULL.dtype)

# =============================================================================
# OOF STACKING
# =============================================================================
def build_oof_base_prior(scores_full_raw, meta_full, sc_clean, Y_SC, n_splits=5, verbose=True):
    groups_full = meta_full["filename"].to_numpy()
    gkf = GroupKFold(n_splits=n_splits)

    oof_base = np.zeros_like(scores_full_raw, dtype=np.float32)
    oof_prior = np.zeros_like(scores_full_raw, dtype=np.float32)
    fold_id = np.full(len(meta_full), -1, dtype=np.int16)

    splits = list(gkf.split(scores_full_raw, groups=groups_full))
    iterator = tqdm(splits, desc="OOF base/prior folds", disable=not verbose)

    for fold, (tr_idx, va_idx) in enumerate(iterator, 1):
        tr_idx = np.sort(tr_idx)
        va_idx = np.sort(va_idx)

        val_files = set(meta_full.iloc[va_idx]["filename"].tolist())
        prior_mask = ~sc_clean["filename"].isin(val_files).values
        prior_df_fold = sc_clean.loc[prior_mask].reset_index(drop=True)
        Y_prior_fold = Y_SC[prior_mask]

        tables = fit_prior_tables(prior_df_fold, Y_prior_fold)

        va_base, va_prior = fuse_scores_with_tables(
            scores_full_raw[va_idx],
            sites=meta_full.iloc[va_idx]["site"].to_numpy(),
            hours=meta_full.iloc[va_idx]["hour_utc"].to_numpy(),
            tables=tables,
        )

        oof_base[va_idx] = va_base
        oof_prior[va_idx] = va_prior
        fold_id[va_idx] = fold

    assert (fold_id >= 0).all()
    return oof_base, oof_prior, fold_id


print("\nBuilding OOF meta-features...")
oof_base, oof_prior, oof_fold_id = build_oof_base_prior(
    scores_full_raw=scores_full_raw,
    meta_full=meta_full,
    sc_clean=sc_clean,
    Y_SC=Y_SC,
    n_splits=5,
    verbose=CFG["verbose"],
)

baseline_oof_auc = macro_auc_skip_empty(Y_FULL, oof_base)
print(f"Honest OOF baseline AUC: {baseline_oof_auc:.6f}")

# =============================================================================
# SELECTIVE SSM MODULE
# =============================================================================
class SelectiveSSM(nn.Module):
    """Simplified Mamba-style selective state space model."""

    def __init__(self, d_model, d_state=16, d_conv=4):
        super().__init__()
        self.d_model = d_model
        self.d_state = d_state

        self.in_proj = nn.Linear(d_model, 2 * d_model, bias=False)

        self.conv1d = nn.Conv1d(
            d_model, d_model, d_conv,
            padding=d_conv - 1, groups=d_model
        )

        self.dt_proj = nn.Linear(d_model, d_model, bias=True)

        A = torch.arange(1, d_state + 1, dtype=torch.float32)
        A = A.unsqueeze(0).expand(d_model, -1)
        self.A_log = nn.Parameter(torch.log(A))

        self.D = nn.Parameter(torch.ones(d_model))

        self.B_proj = nn.Linear(d_model, d_state, bias=False)
        self.C_proj = nn.Linear(d_model, d_state, bias=False)

        self.out_proj = nn.Linear(d_model, d_model, bias=False)

    def forward(self, x):
        B_size, T, D = x.shape

        xz = self.in_proj(x)
        x_ssm, z = xz.chunk(2, dim=-1)

        x_conv = self.conv1d(x_ssm.transpose(1, 2))[:, :, :T].transpose(1, 2)
        x_conv = F.silu(x_conv)

        dt = F.softplus(self.dt_proj(x_conv))
        B_t = self.B_proj(x_conv)
        C_t = self.C_proj(x_conv)
        A = -torch.exp(self.A_log)

        y = self._selective_scan(x_conv, dt, A, B_t, C_t)

        y = y * F.silu(z)
        return self.out_proj(y)

    def _selective_scan(self, x, dt, A, B, C):
        batch, T, D = x.shape
        N = self.d_state

        h = torch.zeros(batch, D, N, device=x.device, dtype=x.dtype)
        ys = []

        for t in range(T):
            dt_t = dt[:, t, :, None]
            dA = torch.exp(A[None] * dt_t)
            dB = dt_t * B[:, t, None, :]
            h = h * dA + x[:, t, :, None] * dB
            y_t = (h * C[:, t, None, :]).sum(-1)
            ys.append(y_t)

        y = torch.stack(ys, dim=1)
        return y + x * self.D[None, None, :]


class ProtoSSMv2(nn.Module):
    """Prototypical State Space Model v2 with metadata awareness."""

    def __init__(self, d_input=1536, d_model=192, d_state=16,
                 n_ssm_layers=2, n_classes=234, n_windows=12,
                 dropout=0.15, n_sites=20, meta_dim=16):
        super().__init__()
        self.d_model = d_model
        self.n_classes = n_classes
        self.n_windows = n_windows

        # Feature projection
        self.input_proj = nn.Sequential(
            nn.Linear(d_input, d_model),
            nn.LayerNorm(d_model),
            nn.GELU(),
            nn.Dropout(dropout),
        )

        # Positional encoding
        self.pos_enc = nn.Parameter(torch.randn(1, n_windows, d_model) * 0.02)

        # Metadata embeddings
        self.site_emb = nn.Embedding(n_sites, meta_dim)
        self.hour_emb = nn.Embedding(24, meta_dim)
        self.meta_proj = nn.Linear(2 * meta_dim, d_model)

        # Bidirectional SSM layers
        self.ssm_fwd = nn.ModuleList()
        self.ssm_bwd = nn.ModuleList()
        self.ssm_merge = nn.ModuleList()
        self.ssm_norm = nn.ModuleList()
        for _ in range(n_ssm_layers):
            self.ssm_fwd.append(SelectiveSSM(d_model, d_state))
            self.ssm_bwd.append(SelectiveSSM(d_model, d_state))
            self.ssm_merge.append(nn.Linear(2 * d_model, d_model))
            self.ssm_norm.append(nn.LayerNorm(d_model))
        self.ssm_drop = nn.Dropout(dropout)

        # Prototypes
        self.prototypes = nn.Parameter(torch.randn(n_classes, d_model) * 0.02)
        self.proto_temp = nn.Parameter(torch.tensor(5.0))

        # Per-class calibration bias
        self.class_bias = nn.Parameter(torch.zeros(n_classes))

        # Per-class gated fusion with Perch logits
        self.fusion_alpha = nn.Parameter(torch.zeros(n_classes))

        # Taxonomic auxiliary head
        self.n_families = 0
        self.family_head = None

    def init_prototypes_from_data(self, embeddings, labels):
        with torch.no_grad():
            h = self.input_proj(embeddings)
            for c in range(self.n_classes):
                mask = labels[:, c] > 0.5
                if mask.sum() > 0:
                    self.prototypes.data[c] = F.normalize(h[mask].mean(0), dim=0)

    def init_family_head(self, n_families, class_to_family):
        self.n_families = n_families
        self.family_head = nn.Linear(self.d_model, n_families)
        self.register_buffer('class_to_family', torch.tensor(class_to_family, dtype=torch.long))

    def forward(self, emb, perch_logits=None, site_ids=None, hours=None):
        B, T, _ = emb.shape

        h = self.input_proj(emb)
        h = h + self.pos_enc[:, :T, :]

        if site_ids is not None and hours is not None:
            s_emb = self.site_emb(site_ids)
            h_emb = self.hour_emb(hours)
            meta = self.meta_proj(torch.cat([s_emb, h_emb], dim=-1))
            h = h + meta[:, None, :]

        for fwd, bwd, merge, norm in zip(
            self.ssm_fwd, self.ssm_bwd, self.ssm_merge, self.ssm_norm
        ):
            residual = h
            h_f = fwd(h)
            h_b = bwd(h.flip(1)).flip(1)
            h = merge(torch.cat([h_f, h_b], dim=-1))
            h = self.ssm_drop(h)
            h = norm(h + residual)

        h_temporal = h

        h_norm = F.normalize(h, dim=-1)
        p_norm = F.normalize(self.prototypes, dim=-1)
        temp = F.softplus(self.proto_temp)
        sim = torch.matmul(h_norm, p_norm.T) * temp + self.class_bias[None, None, :]

        if perch_logits is not None:
            alpha = torch.sigmoid(self.fusion_alpha)[None, None, :]
            species_logits = alpha * sim + (1 - alpha) * perch_logits
        else:
            species_logits = sim

        family_logits = None
        if self.family_head is not None:
            h_pool = h.mean(dim=1)
            family_logits = self.family_head(h_pool)

        return species_logits, family_logits, h_temporal

    def count_parameters(self):
        return sum(p.numel() for p in self.parameters() if p.requires_grad)


print("ProtoSSMv2 architecture defined.")
ssm_cfg = CFG["proto_ssm"]
print(f"Parameter count (d_model={ssm_cfg['d_model']}, 2 layers): "
      f"{ProtoSSMv2(d_model=ssm_cfg['d_model'], n_ssm_layers=2, n_sites=ssm_cfg['n_sites']).count_parameters():,}")

# =============================================================================
# PROTOSM TRAINING FUNCTIONS
# =============================================================================
def build_taxonomy_groups(taxonomy_df, primary_labels):
    for col in ["family", "order", "class_name"]:
        if col in taxonomy_df.columns:
            group_map = taxonomy_df.set_index("primary_label")[col].to_dict()
            break
    else:
        group_map = {label: "Unknown" for label in primary_labels}

    groups = sorted(set(group_map.values()))
    grp_to_idx = {g: i for i, g in enumerate(groups)}
    class_to_group = []
    for label in primary_labels:
        grp = group_map.get(label, "Unknown")
        class_to_group.append(grp_to_idx.get(grp, 0))
    return len(groups), class_to_group, grp_to_idx


def build_site_mapping(meta_df):
    sites = meta_df["site"].unique().tolist()
    site_to_idx = {s: i + 1 for i, s in enumerate(sites)}
    n_sites = len(sites) + 1
    return site_to_idx, n_sites


def reshape_to_files(flat_array, meta_df, n_windows=N_WINDOWS):
    filenames = meta_df["filename"].to_numpy()
    unique_files = []
    seen = set()
    for f in filenames:
        if f not in seen:
            unique_files.append(f)
            seen.add(f)

    n_files = len(unique_files)
    assert len(flat_array) == n_files * n_windows

    new_shape = (n_files, n_windows) + flat_array.shape[1:]
    return flat_array.reshape(new_shape), unique_files


def get_file_metadata(meta_df, file_list, site_to_idx, n_sites_max):
    file_to_row = {}
    filenames = meta_df["filename"].to_numpy()
    sites = meta_df["site"].to_numpy()
    hours = meta_df["hour_utc"].to_numpy()
    for i, f in enumerate(filenames):
        if f not in file_to_row:
            file_to_row[f] = i

    site_ids = np.zeros(len(file_list), dtype=np.int64)
    hour_ids = np.zeros(len(file_list), dtype=np.int64)

    for fi, fname in enumerate(file_list):
        row = file_to_row.get(fname)
        if row is not None:
            sid = site_to_idx.get(sites[row], 0)
            site_ids[fi] = min(sid, n_sites_max - 1)
            hour_ids[fi] = int(hours[row]) % 24

    return site_ids, hour_ids


def train_proto_ssm_single(model, emb_train, logits_train, labels_train,
                           site_ids_train=None, hours_train=None,
                           emb_val=None, logits_val=None, labels_val=None,
                           site_ids_val=None, hours_val=None,
                           file_families_train=None, file_families_val=None,
                           cfg=None, verbose=True):
    if cfg is None:
        cfg = CFG["proto_ssm_train"]

    label_smoothing = cfg.get("label_smoothing", 0.0)

    emb_tr = torch.tensor(emb_train, dtype=torch.float32)
    logits_tr = torch.tensor(logits_train, dtype=torch.float32)
    labels_tr = torch.tensor(labels_train, dtype=torch.float32)

    if label_smoothing > 0:
        labels_tr = labels_tr * (1.0 - label_smoothing) + label_smoothing / 2.0

    site_tr = torch.tensor(site_ids_train, dtype=torch.long) if site_ids_train is not None else None
    hour_tr = torch.tensor(hours_train, dtype=torch.long) if hours_train is not None else None

    has_val = emb_val is not None
    if has_val:
        emb_v = torch.tensor(emb_val, dtype=torch.float32)
        logits_v = torch.tensor(logits_val, dtype=torch.float32)
        labels_v = torch.tensor(labels_val, dtype=torch.float32)
        site_v = torch.tensor(site_ids_val, dtype=torch.long) if site_ids_val is not None else None
        hour_v = torch.tensor(hours_val, dtype=torch.long) if hours_val is not None else None

    fam_tr = torch.tensor(file_families_train, dtype=torch.float32) if file_families_train is not None else None
    fam_v = torch.tensor(file_families_val, dtype=torch.float32) if (has_val and file_families_val is not None) else None

    pos_counts = labels_tr.sum(dim=(0, 1))
    total = labels_tr.shape[0] * labels_tr.shape[1]
    pos_weight = ((total - pos_counts) / (pos_counts + 1)).clamp(max=cfg["pos_weight_cap"])

    optimizer = torch.optim.AdamW(model.parameters(), lr=cfg["lr"], weight_decay=cfg["weight_decay"])
    scheduler = torch.optim.lr_scheduler.OneCycleLR(
        optimizer, max_lr=cfg["lr"],
        epochs=cfg["n_epochs"], steps_per_epoch=1,
        pct_start=0.1, anneal_strategy='cos'
    )

    best_val_loss = float('inf')
    best_state = None
    wait = 0
    history = {"train_loss": [], "val_loss": [], "val_auc": []}

    for epoch in range(cfg["n_epochs"]):
        model.train()
        species_out, family_out, _ = model(emb_tr, logits_tr, site_ids=site_tr, hours=hour_tr)

        loss_bce = F.binary_cross_entropy_with_logits(
            species_out, labels_tr,
            pos_weight=pos_weight[None, None, :]
        )

        loss_distill = F.mse_loss(species_out, logits_tr)

        loss = loss_bce + cfg["distill_weight"] * loss_distill

        if family_out is not None and fam_tr is not None:
            loss_family = F.binary_cross_entropy_with_logits(family_out, fam_tr)
            loss = loss + 0.1 * loss_family

        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()

        model.eval()
        with torch.no_grad():
            if has_val:
                val_out, val_fam, _ = model(emb_v, logits_v, site_ids=site_v, hours=hour_v)
                val_loss = F.binary_cross_entropy_with_logits(
                    val_out, labels_v,
                    pos_weight=pos_weight[None, None, :]
                )

                val_pred = val_out.reshape(-1, val_out.shape[-1]).numpy()
                val_true = labels_v.reshape(-1, labels_v.shape[-1]).numpy()
                try:
                    val_auc = macro_auc_skip_empty(val_true, val_pred)
                except Exception:
                    val_auc = 0.0
            else:
                val_loss = loss
                val_auc = 0.0

        history["train_loss"].append(loss.item())
        history["val_loss"].append(val_loss.item())
        history["val_auc"].append(val_auc)

        if val_loss.item() < best_val_loss:
            best_val_loss = val_loss.item()
            best_state = {k: v.clone() for k, v in model.state_dict().items()}
            wait = 0
        else:
            wait += 1

        if verbose and (epoch + 1) % 20 == 0:
            lr_now = optimizer.param_groups[0]['lr']
            print(f"  Epoch {epoch+1:3d}: train={loss.item():.4f} val={val_loss.item():.4f} "
                  f"auc={val_auc:.4f} lr={lr_now:.6f} wait={wait}")

        if wait >= cfg["patience"]:
            if verbose:
                print(f"  Early stopping at epoch {epoch+1} (best val_loss={best_val_loss:.4f})")
            break

    if best_state is not None:
        model.load_state_dict(best_state)

    if verbose:
        print(f"  Training complete. Best val_loss={best_val_loss:.4f}")
        with torch.no_grad():
            alphas = torch.sigmoid(model.fusion_alpha).numpy()
            print(f"  Fusion alpha: mean={alphas.mean():.3f} min={alphas.min():.3f} max={alphas.max():.3f}")
            print(f"  Proto temperature: {F.softplus(model.proto_temp).item():.3f}")

    return model, history


def run_proto_ssm_oof(emb_files, logits_files, labels_files,
                      site_ids_all, hours_all,
                      file_families, file_groups,
                      n_families, class_to_family,
                      cfg=None, verbose=True):
    if cfg is None:
        cfg = CFG["proto_ssm_train"]

    n_splits = cfg.get("oof_n_splits", 3)
    n_files = len(emb_files)
    ssm_cfg = CFG["proto_ssm"]

    oof_preds = np.zeros((n_files, N_WINDOWS, N_CLASSES), dtype=np.float32)
    fold_histories = []
    fold_alphas = []

    n_unique_groups = len(set(file_groups))
    if n_unique_groups < n_splits:
        print(f"  WARNING: Only {n_unique_groups} groups, reducing n_splits from {n_splits} to {n_unique_groups}")
        n_splits = n_unique_groups
    gkf = GroupKFold(n_splits=n_splits)
    dummy_y = np.zeros(n_files)

    for fold_i, (train_idx, val_idx) in enumerate(gkf.split(dummy_y, dummy_y, file_groups)):
        if verbose:
            print(f"\n--- Fold {fold_i+1}/{n_splits} (train={len(train_idx)}, val={len(val_idx)}) ---")

        fold_model = ProtoSSMv2(
            d_input=emb_files.shape[2],
            d_model=ssm_cfg["d_model"],
            d_state=ssm_cfg["d_state"],
            n_ssm_layers=ssm_cfg["n_ssm_layers"],
            n_classes=N_CLASSES,
            n_windows=N_WINDOWS,
            dropout=ssm_cfg["dropout"],
            n_sites=ssm_cfg["n_sites"],
            meta_dim=ssm_cfg["meta_dim"],
        ).to(DEVICE)

        emb_flat_fold = emb_files[train_idx].reshape(-1, emb_files.shape[2])
        labels_flat_fold = labels_files[train_idx].reshape(-1, N_CLASSES)
        fold_model.init_prototypes_from_data(
            torch.tensor(emb_flat_fold, dtype=torch.float32),
            torch.tensor(labels_flat_fold, dtype=torch.float32)
        )
        fold_model.init_family_head(n_families, class_to_family)

        fold_model, fold_hist = train_proto_ssm_single(
            fold_model,
            emb_files[train_idx], logits_files[train_idx], labels_files[train_idx].astype(np.float32),
            site_ids_train=site_ids_all[train_idx], hours_train=hours_all[train_idx],
            emb_val=emb_files[val_idx], logits_val=logits_files[val_idx],
            labels_val=labels_files[val_idx].astype(np.float32),
            site_ids_val=site_ids_all[val_idx], hours_val=hours_all[val_idx],
            file_families_train=file_families[train_idx],
            file_families_val=file_families[val_idx],
            cfg=cfg, verbose=verbose,
        )

        fold_model.eval()
        with torch.no_grad():
            val_emb = torch.tensor(emb_files[val_idx], dtype=torch.float32)
            val_logits = torch.tensor(logits_files[val_idx], dtype=torch.float32)
            val_sites = torch.tensor(site_ids_all[val_idx], dtype=torch.long)
            val_hours = torch.tensor(hours_all[val_idx], dtype=torch.long)
            val_out, _, _ = fold_model(val_emb, val_logits, site_ids=val_sites, hours=val_hours)
            oof_preds[val_idx] = val_out.numpy()

            fold_alphas.append(torch.sigmoid(fold_model.fusion_alpha).numpy().copy())

        fold_histories.append(fold_hist)

    return oof_preds, fold_histories, fold_alphas


def optimize_ensemble_weight(oof_proto_flat, oof_mlp_flat, y_true_flat):
    weights = np.arange(0.0, 1.05, 0.05)
    results = []

    for w in weights:
        blended = w * oof_proto_flat + (1.0 - w) * oof_mlp_flat
        try:
            auc = macro_auc_skip_empty(y_true_flat, blended)
        except Exception:
            auc = 0.0
        results.append((w, auc))

    best_w, best_auc = max(results, key=lambda x: x[1])
    return best_w, best_auc, results


print("ProtoSSM v2 training functions defined.")

# =============================================================================
# FIT SCALER + PCA
# =============================================================================
print("\n" + "-" * 70)
print("Fitting Scaler + PCA...")
print("-" * 70)

emb_scaler = StandardScaler()
emb_full_scaled = emb_scaler.fit_transform(emb_full)

n_comp = min(CFG["frozen_best_probe"]["pca_dim"], emb_full_scaled.shape[0] - 1, emb_full_scaled.shape[1])
emb_pca = PCA(n_components=n_comp)
Z_FULL = emb_pca.fit_transform(emb_full_scaled).astype(np.float32)

print(f"PCA components: {n_comp}")
print(f"Explained variance: {emb_pca.explained_variance_ratio_.sum():.4f}")

# =============================================================================
# TRAIN MLP PROBES
# =============================================================================
print("\n" + "-" * 70)
print("Training MLP Probes...")
print("-" * 70)

PROBE_CLASS_IDX = np.where(Y_FULL.sum(axis=0) >= CFG["frozen_best_probe"]["min_pos"])[0].astype(np.int32)

mlp_probes = {}
for cls_idx in tqdm(PROBE_CLASS_IDX, desc="Training MLP probes", disable=not CFG["verbose"]):
    y = Y_FULL[:, cls_idx]
    if y.sum() == 0 or y.sum() == len(y):
        continue

    X_cls = build_class_features(
        Z_FULL,
        raw_col=scores_full_raw[:, cls_idx],
        prior_col=oof_prior[:, cls_idx],
        base_col=oof_base[:, cls_idx],
    )

    n_pos = int(y.sum())
    n_neg = len(y) - n_pos
    if n_pos > 0 and n_neg > n_pos:
        repeat = max(1, n_neg // n_pos)
        pos_idx = np.where(y == 1)[0]
        X_bal = np.vstack([X_cls, np.tile(X_cls[pos_idx], (repeat, 1))])
        y_bal = np.concatenate([y, np.ones(len(pos_idx) * repeat, dtype=y.dtype)])
    else:
        X_bal, y_bal = X_cls, y

    clf = MLPClassifier(**CFG["mlp_params"])
    clf.fit(X_bal, y_bal)
    mlp_probes[cls_idx] = clf

print(f"MLP probes trained: {len(mlp_probes)}")

# =============================================================================
# EVALUATE OOF MLP PROBES
# =============================================================================
print("\nEvaluating OOF MLP probes...")

oof_mlp = oof_base.copy()
for cls_idx in tqdm(mlp_probes.keys(), desc="Applying MLP probes"):
    X_cls = build_class_features(
        Z_FULL,
        raw_col=scores_full_raw[:, cls_idx],
        prior_col=oof_prior[:, cls_idx],
        base_col=oof_base[:, cls_idx],
    )
    prob = mlp_probes[cls_idx].predict_proba(X_cls)[:, 1].astype(np.float32)
    pred = np.log(prob + 1e-7) - np.log(1 - prob + 1e-7)
    oof_mlp[:, cls_idx] = (1.0 - CFG["frozen_best_probe"]["alpha"]) * oof_base[:, cls_idx] + CFG["frozen_best_probe"]["alpha"] * pred

mlp_auc = macro_auc_skip_empty(Y_FULL, oof_mlp)
print(f"MLP-only OOF AUC: {mlp_auc:.4f}")

# =============================================================================
# TRAIN PROTOSM V2
# =============================================================================
print("\n" + "-" * 70)
print("Training ProtoSSM v2...")
print("-" * 70)

# Reshape to file-level
emb_files, file_list = reshape_to_files(emb_full, meta_full)
logits_files, _ = reshape_to_files(scores_full_raw, meta_full)
labels_files, _ = reshape_to_files(Y_FULL, meta_full)

print(f"Reshaped to file-level: emb={emb_files.shape}, logits={logits_files.shape}, labels={labels_files.shape}")

# Build taxonomy groups, site mapping
n_families, class_to_family, fam_to_idx = build_taxonomy_groups(taxonomy, PRIMARY_LABELS)
site_to_idx, n_sites_mapped = build_site_mapping(meta_full)
n_sites_cfg = CFG["proto_ssm"]["n_sites"]

site_ids_all, hours_all = get_file_metadata(meta_full, file_list, site_to_idx, n_sites_cfg)

# Build per-file family labels
file_families = np.zeros((len(file_list), n_families), dtype=np.float32)
for fi in range(len(file_list)):
    active_classes = np.where(labels_files[fi].sum(axis=0) > 0)[0]
    for ci in active_classes:
        file_families[fi, class_to_family[ci]] = 1.0

# OOF Cross-Validation
ENSEMBLE_WEIGHT_PROTO = 0.5
oof_proto_flat = None
fold_alphas = []

file_groups = np.array([f.split("_")[3] if len(f.split("_")) > 3 else f for f in file_list])
print(f"File groups for OOF: {len(set(file_groups))} unique groups")

t0_oof = time.time()
oof_proto_preds, fold_histories, fold_alphas = run_proto_ssm_oof(
    emb_files, logits_files, labels_files,
    site_ids_all, hours_all,
    file_families, file_groups,
    n_families, class_to_family,
    cfg=CFG["proto_ssm_train"],
    verbose=CFG["verbose"],
)
oof_time = time.time() - t0_oof
print(f"\nOOF cross-validation time: {oof_time:.1f}s")

oof_proto_flat = oof_proto_preds.reshape(-1, N_CLASSES)
y_flat = labels_files.reshape(-1, N_CLASSES).astype(np.float32)

overall_oof_auc_proto = macro_auc_skip_empty(y_flat, oof_proto_flat)
print(f"ProtoSSM OOF macro AUC: {overall_oof_auc_proto:.4f}")

LOGS["oof_auc_proto"] = overall_oof_auc_proto
LOGS["oof_time"] = oof_time

# Train final model on ALL data
ssm_cfg = CFG["proto_ssm"]
model = ProtoSSMv2(
    d_input=emb_full.shape[1],
    d_model=ssm_cfg["d_model"],
    d_state=ssm_cfg["d_state"],
    n_ssm_layers=ssm_cfg["n_ssm_layers"],
    n_classes=N_CLASSES,
    n_windows=N_WINDOWS,
    dropout=ssm_cfg["dropout"],
    n_sites=ssm_cfg["n_sites"],
    meta_dim=ssm_cfg["meta_dim"],
).to(DEVICE)

emb_flat_tensor = torch.tensor(emb_full, dtype=torch.float32)
labels_flat_tensor = torch.tensor(Y_FULL, dtype=torch.float32)
model.init_prototypes_from_data(emb_flat_tensor, labels_flat_tensor)
model.init_family_head(n_families, class_to_family)

print(f"\nProtoSSM v2 parameters: {model.count_parameters():,}")

t0_final = time.time()
model, train_history = train_proto_ssm_single(
    model,
    emb_files, logits_files, labels_files.astype(np.float32),
    site_ids_train=site_ids_all, hours_train=hours_all,
    cfg=CFG["proto_ssm_train"],
    verbose=True,
)
train_time = time.time() - t0_final
print(f"Final model training time: {train_time:.1f}s")

# =============================================================================
# OPTIMIZE ENSEMBLE WEIGHT
# =============================================================================
print("\n" + "-" * 70)
print("Optimizing ensemble weight...")
print("-" * 70)

best_w, best_auc, weight_results = optimize_ensemble_weight(oof_proto_flat, oof_mlp, y_flat)
ENSEMBLE_WEIGHT_PROTO = best_w

print(f"Best ProtoSSM weight: {ENSEMBLE_WEIGHT_PROTO:.2f}")
print(f"Best ensemble OOF AUC: {best_auc:.4f}")
print(f"MLP-only OOF AUC: {mlp_auc:.4f}")

LOGS["ensemble_weight"] = ENSEMBLE_WEIGHT_PROTO
LOGS["ensemble_auc"] = best_auc
LOGS["mlp_only_auc"] = mlp_auc

# =============================================================================
# RESIDUAL SSM
# =============================================================================
print("\n" + "-" * 70)
print("Training ResidualSSM...")
print("-" * 70)


class ResidualSSM(nn.Module):
    """Lightweight SSM for second-pass error correction."""

    def __init__(self, d_input=1536, d_scores=234, d_model=64, d_state=8,
                 n_classes=234, n_windows=12, dropout=0.1, n_sites=20):
        super().__init__()
        self.d_model = d_model
        self.n_classes = n_classes

        self.input_proj = nn.Sequential(
            nn.Linear(d_input + d_scores, d_model),
            nn.LayerNorm(d_model),
            nn.GELU(),
            nn.Dropout(dropout),
        )

        self.site_emb = nn.Embedding(n_sites, 8)
        self.hour_emb = nn.Embedding(24, 8)
        self.meta_proj = nn.Linear(16, d_model)

        self.pos_enc = nn.Parameter(torch.randn(1, n_windows, d_model) * 0.02)

        self.ssm_fwd = SelectiveSSM(d_model, d_state)
        self.ssm_bwd = SelectiveSSM(d_model, d_state)
        self.ssm_merge = nn.Linear(2 * d_model, d_model)
        self.ssm_norm = nn.LayerNorm(d_model)
        self.ssm_drop = nn.Dropout(dropout)

        self.output_head = nn.Linear(d_model, n_classes)

        nn.init.zeros_(self.output_head.weight)
        nn.init.zeros_(self.output_head.bias)

    def forward(self, emb, first_pass_scores, site_ids=None, hours=None):
        B, T, _ = emb.shape

        x = torch.cat([emb, first_pass_scores], dim=-1)
        h = self.input_proj(x)

        if site_ids is not None and hours is not None:
            site_e = self.site_emb(site_ids.clamp(0, self.site_emb.num_embeddings - 1))
            hour_e = self.hour_emb(hours.clamp(0, 23))
            meta = self.meta_proj(torch.cat([site_e, hour_e], dim=-1))
            h = h + meta.unsqueeze(1)

        h = h + self.pos_enc[:, :T, :]

        residual = h
        h_f = self.ssm_fwd(h)
        h_b = self.ssm_bwd(h.flip(1)).flip(1)
        h = self.ssm_merge(torch.cat([h_f, h_b], dim=-1))
        h = self.ssm_drop(h)
        h = self.ssm_norm(h + residual)

        return self.output_head(h)

    def count_parameters(self):
        return sum(p.numel() for p in self.parameters() if p.requires_grad)


# Compute first-pass scores on training data
model.eval()
with torch.no_grad():
    emb_train_t = torch.tensor(emb_files, dtype=torch.float32)
    logits_train_t = torch.tensor(logits_files, dtype=torch.float32)
    site_train_t = torch.tensor(site_ids_all, dtype=torch.long)
    hour_train_t = torch.tensor(hours_all, dtype=torch.long)

    proto_train_out, _, _ = model(emb_train_t, logits_train_t, site_ids=site_train_t, hours=hour_train_t)
    proto_train_scores = proto_train_out.numpy()

# MLP probe scores
mlp_train_scores_flat = oof_mlp.copy()

first_pass_files = (
    ENSEMBLE_WEIGHT_PROTO * proto_train_scores +
    (1 - ENSEMBLE_WEIGHT_PROTO) * mlp_train_scores_flat.reshape(-1, N_WINDOWS, N_CLASSES)
).astype(np.float32)

# Compute residuals
labels_float = labels_files.astype(np.float32)
first_pass_probs = 1.0 / (1.0 + np.exp(-first_pass_files))
residuals = labels_float - first_pass_probs

print(f"First-pass training scores: {first_pass_files.shape}")
print(f"Residuals: mean={residuals.mean():.4f}, std={residuals.std():.4f}")

# Train ResidualSSM
res_cfg = CFG["residual_ssm"]
res_model = ResidualSSM(
    d_input=emb_full.shape[1],
    d_scores=N_CLASSES,
    d_model=res_cfg["d_model"],
    d_state=res_cfg["d_state"],
    n_classes=N_CLASSES,
    n_windows=N_WINDOWS,
    dropout=res_cfg["dropout"],
    n_sites=CFG["proto_ssm"]["n_sites"],
).to(DEVICE)

print(f"ResidualSSM parameters: {res_model.count_parameters():,}")

n_files = len(file_list)
n_val = max(1, int(n_files * 0.15))
perm = np.random.RandomState(123).permutation(n_files)
val_i = perm[:n_val]
train_i = perm[n_val:]

emb_tr = torch.tensor(emb_files[train_i], dtype=torch.float32)
fp_tr = torch.tensor(first_pass_files[train_i], dtype=torch.float32)
res_tr = torch.tensor(residuals[train_i], dtype=torch.float32)
site_tr = torch.tensor(site_ids_all[train_i], dtype=torch.long)
hour_tr = torch.tensor(hours_all[train_i], dtype=torch.long)

emb_va = torch.tensor(emb_files[val_i], dtype=torch.float32)
fp_va = torch.tensor(first_pass_files[val_i], dtype=torch.float32)
res_va = torch.tensor(residuals[val_i], dtype=torch.float32)
site_va = torch.tensor(site_ids_all[val_i], dtype=torch.long)
hour_va = torch.tensor(hours_all[val_i], dtype=torch.long)

optimizer = torch.optim.AdamW(res_model.parameters(), lr=res_cfg["lr"], weight_decay=1e-3)
scheduler = torch.optim.lr_scheduler.OneCycleLR(
    optimizer, max_lr=res_cfg["lr"],
    epochs=res_cfg["n_epochs"], steps_per_epoch=1,
    pct_start=0.1, anneal_strategy='cos'
)

best_val_loss = float('inf')
best_state = None
wait = 0

t0_res = time.time()
for epoch in range(res_cfg["n_epochs"]):
    res_model.train()
    correction = res_model(emb_tr, fp_tr, site_ids=site_tr, hours=hour_tr)
    loss = F.mse_loss(correction, res_tr)

    optimizer.zero_grad()
    loss.backward()
    torch.nn.utils.clip_grad_norm_(res_model.parameters(), 1.0)
    optimizer.step()
    scheduler.step()

    res_model.eval()
    with torch.no_grad():
        val_corr = res_model(emb_va, fp_va, site_ids=site_va, hours=hour_va)
        val_loss = F.mse_loss(val_corr, res_va)

    if val_loss.item() < best_val_loss:
        best_val_loss = val_loss.item()
        best_state = {k: v.clone() for k, v in res_model.state_dict().items()}
        wait = 0
    else:
        wait += 1

    if (epoch + 1) % 10 == 0:
        print(f"  ResidualSSM epoch {epoch+1}: train={loss.item():.6f} val={val_loss.item():.6f} wait={wait}")

    if wait >= res_cfg["patience"]:
        print(f"  ResidualSSM early stop at epoch {epoch+1}")
        break

if best_state is not None:
    res_model.load_state_dict(best_state)

res_time = time.time() - t0_res
print(f"ResidualSSM training time: {res_time:.1f}s")
print(f"Best val MSE: {best_val_loss:.6f}")

CORRECTION_WEIGHT = res_cfg["correction_weight"]
print(f"Correction weight: {CORRECTION_WEIGHT}")

LOGS["residual_ssm"] = {
    "params": res_model.count_parameters(),
    "train_time": res_time,
    "best_val_mse": best_val_loss,
    "correction_weight": CORRECTION_WEIGHT,
}

# =============================================================================
# SAVE OUTPUT FILES
# =============================================================================
print("\n" + "-" * 70)
print("Saving Output Files...")
print("-" * 70)


def serialize_pickle(obj):
    buf = BytesIO()
    pickle.dump(obj, buf)
    return np.frombuffer(buf.getvalue(), dtype=np.uint8)


# ProtoSSM state
protossm_state = {k: v.numpy() for k, v in model.state_dict().items()}

# ResidualSSM state
residual_state = {k: v.numpy() for k, v in res_model.state_dict().items()}

# Prior tables
final_prior_tables = fit_prior_tables(sc_clean.reset_index(drop=True), Y_SC)

# Build save dict
save_dict = {
    # Embeddings and scores
    "emb_full": emb_full,
    "scores_full_raw": scores_full_raw,
    "oof_base": oof_base,
    "oof_prior": oof_prior,

    # Preprocessing
    "scaler_mean": emb_scaler.mean_.astype(np.float32),
    "scaler_scale": emb_scaler.scale_.astype(np.float32),
    "pca_components": emb_pca.components_.astype(np.float32),
    "pca_mean": emb_pca.mean_.astype(np.float32),

    # Index arrays
    "bc_indices": BC_INDICES,
    "mapped_pos": MAPPED_POS,
    "unmapped_pos": UNMAPPED_POS,
    "mapped_bc_indices": MAPPED_BC_INDICES,
    "selected_proxy_pos": selected_proxy_pos,
    "idx_active_texture": idx_active_texture,
    "idx_active_event": idx_active_event,
    "idx_mapped_active_texture": idx_mapped_active_texture,
    "idx_mapped_active_event": idx_mapped_active_event,
    "idx_selected_proxy_active_texture": idx_selected_proxy_active_texture,
    "idx_selected_prioronly_active_texture": idx_selected_prioronly_active_texture,
    "idx_selected_prioronly_active_event": idx_selected_prioronly_active_event,
    "idx_unmapped_inactive": idx_unmapped_inactive,

    # Prior tables
    "global_p": final_prior_tables["global_p"],
    "site_p": final_prior_tables["site_p"],
    "site_n": final_prior_tables["site_n"],
    "hour_p": final_prior_tables["hour_p"],
    "hour_n": final_prior_tables["hour_n"],
    "sh_p": final_prior_tables["sh_p"],
    "sh_n": final_prior_tables["sh_n"],

    # Probes
    "mlp_probes_bytes": serialize_pickle(mlp_probes),
    "probe_classes": PROBE_CLASS_IDX,

    # Config
    "config": np.array(json.dumps({
        "n_classes": N_CLASSES,
        "n_windows": N_WINDOWS,
        "ensemble_weight": ENSEMBLE_WEIGHT_PROTO,
        "correction_weight": CORRECTION_WEIGHT,
        "best_fusion": CFG["best_fusion"],
        "frozen_best_probe": CFG["frozen_best_probe"],
    })),
    "primary_labels": np.array(PRIMARY_LABELS, dtype=object),
}

# ProtoSSM state
for k, v in protossm_state.items():
    save_dict[f"protossm_{k}"] = v

# ResidualSSM state
for k, v in residual_state.items():
    save_dict[f"residual_{k}"] = v

# Site/hour mappings
save_dict["site_to_i"] = np.array(list(final_prior_tables["site_to_i"].items()), dtype=object)
save_dict["hour_to_i"] = np.array(list(final_prior_tables["hour_to_i"].items()), dtype=object)

# Proxy mapping
save_dict["proxy_map_keys"] = np.array(list(selected_proxy_pos_to_bc.keys()), dtype=np.int32)
save_dict["proxy_map_vals"] = np.array([selected_proxy_pos_to_bc[k] for k in selected_proxy_pos_to_bc.keys()], dtype=object)

# Save npz
np.savez_compressed(OUTPUT_DIR / "full_perch_arrays.npz", **save_dict)

# Save metadata
meta_out = meta_full[["row_id", "filename", "site", "hour_utc"]].copy()
meta_out["n_classes"] = N_CLASSES
meta_out["n_windows"] = N_WINDOWS
meta_out["ensemble_weight"] = ENSEMBLE_WEIGHT_PROTO
meta_out["correction_weight"] = CORRECTION_WEIGHT
meta_out["primary_labels"] = json.dumps(PRIMARY_LABELS)

meta_out.to_parquet(OUTPUT_DIR / "full_perch_meta.parquet", index=False)

# =============================================================================
# FINAL SUMMARY
# =============================================================================
wall_time = time.time() - _WALL_START

print("\n" + "=" * 70)
print("Training Complete!")
print("=" * 70)

print(f"\nProtoSSM OOF AUC: {LOGS.get('oof_auc_proto', 0):.4f}")
print(f"MLP-only OOF AUC: {LOGS.get('mlp_only_auc', 0):.4f}")
print(f"Ensemble OOF AUC: {LOGS.get('ensemble_auc', 0):.4f}")
print(f"Optimized ProtoSSM weight: {ENSEMBLE_WEIGHT_PROTO:.2f}")
print(f"Correction weight: {CORRECTION_WEIGHT:.2f}")
print(f"Wall time: {wall_time:.1f}s")

print(f"\nOutput Files:")
print(f"  - full_perch_arrays.npz")
print(f"  - full_perch_meta.parquet")